# 01 — Data Generation & Validation

Builds the calibrated individual-level DGP and runs the three-layer
validation suite (aggregation recovery, sampling-noise sanity check,
placebo check). See `data_generation.py` and `validation.py` for full
docstrings, including the two bugs the validation process caught and
fixed (cross-tier interpolation smearing, and the no_voucher/$0-copay
anchor conflation).

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from data_generation import VoucherDGP
from validation import (
    aggregation_recovery_check,
    sampling_noise_check,
    placebo_check,
    TOLERANCE_PCT,
)

## Generate the individual-level experiment log

In [2]:
dgp = VoucherDGP(calibration_path="../data/raw_benchmarks/case_summary_tables.csv")
experiment_log = dgp.simulate_users()
experiment_log.to_csv("../data/processed/experiment_log.csv", index=False)
print(f"Simulated {len(experiment_log):,} users")
experiment_log.head()

Simulated 200,000 users


,user_id,tier,usage_rate,condition,copay_usd,min_spend_usd,voucher_count,ordered,profit
0,0,90-99,0.9336,s19_m199_c3,19.0,199.0,3,0,0.0
1,1,30-69,0.3373,s0_m249_c3,0.0,249.0,3,0,0.0
2,2,100,1.0000,s0_m249_c3,0.0,249.0,3,0,0.0
3,3,80-89,0.8572,s9_m249_c3,9.0,249.0,3,0,0.0
4,4,0-29,0.1171,no_voucher,0.0,0.0,0,0,0.0


## Check 1 — Aggregation recovery vs. the 7 originally-tested conditions

Simulates at large N (low noise) and compares tier x condition aggregates
against the transcribed PDF numbers. Cells outside +/-15% are flagged.

In [3]:
recovery = aggregation_recovery_check(dgp)
recovery[["tier", "condition", "target_order", "order_per_user", "order_pct_error"]]

,tier,condition,target_order,order_per_user,order_pct_error
0,0-29,no_voucher,0.082,0.0853,3.9675
1,0-29,s0_m199_c1,0.084,0.0788,-6.1646
2,0-29,s0_m249_c3,0.082,0.0829,1.0821
3,0-29,s0_m299_c3,0.083,0.0825,-0.5969
4,0-29,s19_m199_c1,0.083,0.0779,-6.1948
5,0-29,s19_m199_c3,0.084,0.0800,-4.8097
6,0-29,s9_m249_c3,0.082,0.0801,-2.3076
7,100,no_voucher,0.031,0.0309,-0.1716
8,100,s0_m199_c1,0.039,0.0355,-8.8765
9,100,s0_m249_c3,0.036,0.0369,2.6076


In [4]:
n_fail_order = (~recovery["order_within_tolerance"]).sum()
n_fail_profit = (~recovery["profit_within_tolerance"]).sum()
print(f"Order cells outside +/-{TOLERANCE_PCT}%: {n_fail_order} / {len(recovery)}")
print(f"Profit cells outside +/-{TOLERANCE_PCT}%: {n_fail_profit} / {len(recovery)}")

Order cells outside +/-15.0%: 1 / 42
Profit cells outside +/-15.0%: 6 / 42


Residual misses concentrate in the 90-99% tier under stacked
copay+count extrapolation — see DGP_ASSUMPTIONS.md for why this is a
plausible real copay x count interaction, not a bug.

## Check 2 — Sampling-noise sanity check

A DGP that reproduces the target to 4 decimal places on every repeat at
realistic N would itself be a red flag (real data is noisy).

In [5]:
noise = sampling_noise_check(dgp, n_users=20_000)
print(noise[["order_rate"]].describe())
print(f"Empirical SD: {noise['empirical_sd'].iloc[0]:.5f}")
print(f"Theoretical binomial SE: {noise['theoretical_binomial_se'].iloc[0]:.5f}")

       order_rate
count   20.000000
mean     0.026846
std      0.007200
min      0.016100
25%      0.021277
50%      0.027357
75%      0.031743
max      0.042328
Empirical SD: 0.00702
Theoretical binomial SE: 0.00671


## Check 3 — Placebo check

Shuffling condition labels should collapse the apparent treatment effect.

In [6]:
placebo = placebo_check(dgp)
placebo

,real_order_rate,placebo_order_rate,real_range,placebo_range
condition,,,,
no_voucher,0.075850,0.080932,0.005198,0.004839
s0_m199_c1,0.080149,0.080289,0.005198,0.004839
s0_m249_c3,0.080415,0.078366,0.005198,0.004839
s0_m299_c3,0.080206,0.078464,0.005198,0.004839
s19_m199_c1,0.079942,0.080329,0.005198,0.004839
s19_m199_c3,0.076364,0.079573,0.005198,0.004839
s9_m249_c3,0.081048,0.076094,0.005198,0.004839
